This post captures my notes for setting up the git CLI to use my AWS identity to clone [CodeCommit](https://aws.amazon.com/codecommit/) repositories.

You can set up how git provides [authorization credentials](https://git-scm.com/docs/gitcredentials) in the git config. For example, if you want to define a  handle for authenticating with a specific URL, you can add something like the following to your git config:

```toml
[credential "https://some.hosting.service"]
  username = janedoe
```

Git also lets you use "helpers." Helpers are external credential providers. To use the AWS CLI as a credential helper, you would define the helper as a shell snippet where the AWS CLI is called such that it returns credentials to authenticate with CodeCommit. The AWS CLI command to retrieve credentials looks like this:

```bash
printf "protocol=https\n host=git-codecommit.us-east-1.amazonaws.com\n path=/v1/repos/some-repo-id" \ # <1>
  | aws codecommit credential-helper get
```
1. line delimited repo protocol/host/path info.

The command above ⬆️ returns a username/password.

To configure git to use the AWS codecommit credential helper. You modify the git config to direct git to run a shell snippet to retrieve authorization credentials:

```toml
[credential] # <1>
  helper = !aws codecommit credential-helper $@
  useHttpPath = true # <2>
```
1. You can optionally provide a full URL to a specific repo here (see above).
2. You need to set this to true because the `aws codecommit credential helper` command expects the host *and* path repo info. 

::: {.callout-note}
You can also use git to modify the config:

```bash
git config --global credential.helper '!aws codecommit credential-helper $@'
git config --global credential.UseHttpPath true
```

:::

And now you can interact with CodeCommit repos with git! 

::: {.callout-note}
This is a helpful command to find the clone URL for a CodeCommit repo:
```bash
aws codecommit get-repository --repository-name some-repo-name \
  | jq -r '.repositoryMetadata.cloneUrlHttp' \
  | pbcopy
```

:::